# Qualidade de Dados

Auditoria nas cinco dimensões exigidas: Completude, Consistência, Unicidade, Acurácia e
Outliers. Cada seção imprime números que devem ser transcritos para a Seção 5 do README.

In [0]:
from pyspark.sql import functions as F

# Parâmetro: usado pelo Job (Jobs & Pipelines) e com padrão para execução interativa.
dbutils.widgets.text("catalog", "workspace")

CATALOG = dbutils.widgets.get("catalog")

spark.sql(f"USE CATALOG {CATALOG}")

TABELAS_GOLD = [
    "dim_municipio",
    "fato_desempenho_educacional",
    "fato_infraestrutura_escolar",
    "fato_investimento_social",
]

## 1. Completude

Proporção de nulos por coluna. O caso esperado é o IDEB: municípios muito pequenos não
ofertam todas as etapas de ensino, então ausência de nota é informação legítima, não falha
de carga.

In [0]:
for tabela in TABELAS_GOLD:
    df = spark.table(f"{CATALOG}.gold.{tabela}")
    total = df.count()
    nulos = df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns
    ]).collect()[0].asDict()

    incompletas = {c: n for c, n in nulos.items() if n}
    print(f"\n{tabela} ({total:,} linhas)")
    if not incompletas:
        print("  sem nulos")
    for coluna, qtd in sorted(incompletas.items(), key=lambda x: -x[1]):
        print(f"  {coluna:34} {qtd:>7,} nulos ({qtd/total:6.2%})")

## 2. Consistência

Os valores respeitam os domínios esperados: UF entre as 27 siglas oficiais, IDEB na escala
de 0 a 10 e percentuais de infraestrutura entre 0 e 100.

In [0]:
UFS_VALIDAS = ["AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG","PA","PB",
               "PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"]

uf_invalida = spark.table(f"{CATALOG}.gold.dim_municipio") \
    .where(~F.col("sigla_uf").isin(UFS_VALIDAS)).count()

ideb_fora_escala = spark.table(f"{CATALOG}.gold.fato_desempenho_educacional") \
    .where((F.col("vl_ideb") < 0) | (F.col("vl_ideb") > 10)).count()

df_infra = spark.table(f"{CATALOG}.gold.fato_infraestrutura_escolar")
colunas_pct = [c for c in df_infra.columns if c.startswith("pct_")]
pct_fora_faixa = df_infra.where(
    F.greatest(*[F.when((F.col(c) < 0) | (F.col(c) > 100), 1).otherwise(0) for c in colunas_pct]) == 1
).count()

print(f"UFs fora das 27 oficiais:                 {uf_invalida}")
print(f"Notas de IDEB fora da escala [0, 10]:     {ideb_fora_escala}")
print(f"Percentuais de infra fora de [0, 100]:    {pct_fora_faixa}")

## 3. Unicidade

A chave de cada fato não pode se repetir. Esta é a checagem que pega duplicação silenciosa
vinda de junção mal feita.

In [0]:
CHAVES = {
    "dim_municipio": ["codigo_municipio_ibge"],
    "fato_desempenho_educacional": ["codigo_municipio_ibge", "ano", "etapa_ensino"],
    "fato_infraestrutura_escolar": ["codigo_municipio_ibge", "ano"],
    "fato_investimento_social": ["codigo_municipio_ibge", "ano"],
}

for tabela, chave in CHAVES.items():
    df = spark.table(f"{CATALOG}.gold.{tabela}")
    duplicadas = df.groupBy(*chave).count().where("count > 1").count()
    assert duplicadas == 0, f"{tabela}: {duplicadas} chaves duplicadas"
    print(f"{tabela:32} chave {str(chave):58} 0 duplicatas - OK")

## 4. Acurácia

Duas verificações independentes: integridade referencial contra a dimensão, e
reconciliação entre camadas, conferindo que os totais que entraram na Bronze chegam à
Gold sem perda nem duplicação.

In [0]:
import math

dim_codigos = {
    l["codigo_municipio_ibge"]
    for l in spark.table(f"{CATALOG}.gold.dim_municipio").select("codigo_municipio_ibge").collect()
}

for fato in TABELAS_GOLD[1:]:
    codigos = {
        l["codigo_municipio_ibge"]
        for l in spark.table(f"{CATALOG}.gold.{fato}").select("codigo_municipio_ibge").distinct().collect()
    }
    orfaos = len(codigos - dim_codigos)
    assert orfaos == 0, f"{fato}: {orfaos} códigos sem correspondência na dimensão"
    print(f"{fato:32} 0 códigos sem correspondência na dimensão - OK")


def total(tabela, expressao):
    return spark.table(f"{CATALOG}.{tabela}").agg(expressao).first()[0]


# escolas públicas em atividade: mesmo filtro da Silver, aplicado direto na Bronze
escolas_bronze = spark.table(f"{CATALOG}.bronze.censo_escolar_escolas").where(
    (F.col("TP_SITUACAO_FUNCIONAMENTO") == 1) & (F.col("TP_DEPENDENCIA").isin(1, 2, 3))
).count()

RECONCILIACOES = [
    ("Bolsa Família - valor total",
     total("bronze.bolsa_familia_municipio", F.sum("valor_total")),
     total("gold.fato_investimento_social", F.sum("valor_total"))),
    ("Bolsa Família - benefícios",
     total("bronze.bolsa_familia_municipio", F.sum("quantidade_beneficiados")),
     total("gold.fato_investimento_social", F.sum("quantidade_beneficiados"))),
    ("População",
     total("bronze.populacao_municipios", F.sum("populacao")),
     total("gold.dim_municipio", F.sum("populacao"))),
    ("Escolas públicas em atividade",
     escolas_bronze,
     total("gold.fato_infraestrutura_escolar", F.sum("qt_escolas_publicas"))),
]

print()
for nome, bronze, gold in RECONCILIACOES:
    assert math.isclose(bronze, gold, rel_tol=1e-9), f"{nome}: bronze {bronze:,} x gold {gold:,}"
    print(f"{nome:32} bronze {bronze:>20,.2f} = gold - OK")

## 5. Outliers

Regra do IQR sobre o Bolsa Família per capita. Valores extremos aqui tendem a ser
municípios pequenos e muito vulneráveis - informação real, não erro. Por isso são
identificados e mantidos, não removidos.

In [0]:
df_investimento = spark.table(f"{CATALOG}.gold.fato_investimento_social")
q1, q3 = df_investimento.approxQuantile("valor_per_capita", [0.25, 0.75], 0.01)
iqr = q3 - q1
limite_inferior, limite_superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr

outliers = df_investimento.where(
    (F.col("valor_per_capita") < limite_inferior) | (F.col("valor_per_capita") > limite_superior)
)

print(f"Q1 = {q1:.2f} | Q3 = {q3:.2f} | IQR = {iqr:.2f}")
print(f"Limites: [{limite_inferior:.2f}, {limite_superior:.2f}]")
print(f"Outliers: {outliers.count()} municípios ({outliers.count()/df_investimento.count():.1%})")

display(
    outliers.join(spark.table(f"{CATALOG}.gold.dim_municipio"), "codigo_municipio_ibge")
    .select("nome_municipio", "sigla_uf", "populacao", "valor_per_capita", "taxa_cobertura_pct")
    .orderBy(F.desc("valor_per_capita"))
    .limit(15)
)